# D4-07 Optional Trails - premise and imported LCI example

**Authors**: Romain Sacchi (PSI)

**Contact**: romain.sacchi@psi.ch

## Purpose

This notebook demonstrates a realistic workflow that combines a **Premise data package**
with **user‑provided inventories** (Excel) and then runs temporal LCA, static LCA,
and climate‑emulator post‑processing (radiative forcing + temperature).

You will learn how to:

1. Load a Premise data package and reuse cached matrices.
2. Inspect the technosphere (`A`) and biosphere (`B`) matrices.
3. Select LCIA methods.
4. Import an Excel inventory and see how it integrates.
5. Run temporal routing, temporal LCA, and static LCA.
6. Plot temporal scores and link results to FaIR.

## Theory (short)

TRAILS (Temporal Routing And Aggregation of Impacts across Life‑cycle Systems)
extends classic LCA by making **time an explicit dimension**. Exchanges can carry
**temporal distributions** (discrete, normal, lognormal, uniform, triangular),
which are expanded into year offsets during traversal. For each calendar year
that becomes active, the system is solved and biosphere flows are accumulated
at their respective years. This yields **time‑resolved inventories** and **impact
scores** that answer questions like *when impacts occur* and *which upstream
suppliers dominate over time*.

**Environment requirements (from `pyproject.toml`):**
- Python `>=3.10, <3.13`
- Dependencies are loaded from `requirements.txt`
- Optional extras: `testing` (pytest), `docs` (sphinx)

**Prerequisites**
- A Premise data package (zip or directory)
- An Excel inventory file (example: `lci-pass_cars.xlsx`)


In [ ]:
import logging
from pathlib import Path
from datapackage import Package

from trails import (
    Trails,
    get_lcia_method_names,
    plot_temporal_scores,
    plot_rf,
    plot_temp,
    search_activity,
    clear_cache,
)
from trails.logging import configure_trails_logging


## 1. Load the Premise data package

Update the path below to your Premise export. The first run will build and cache
matrix interpolations; later runs reuse the cache. Use `clear_cache()` if you
change the package and want a clean rebuild.


In [ ]:
# Optional: clear cache to force rebuild
# clear_cache()

configure_trails_logging(file_level=logging.INFO)

# Update to your Premise package path
premise_path = Path('export/datapackage/paris-trails-datapackage.zip')
package = Package(str(premise_path))

trails = Trails(package, interpolate_annual=True, debug=False)


## 2. Explore matrices and indices (A, B)

TRAILS exposes the technosphere matrix `A` and biosphere matrix `B` as 3D arrays:

- `A` indexed by `(year, activity, activity)`
- `B` indexed by `(year, activity, flow)`

These matrices are cached on first load. If you change the data package and
need a clean rebuild, call `clear_cache()` and re‑initialize `Trails`.


In [ ]:
# Inspect matrix shapes (year, activity, activity) and (year, activity, flow)
print("A shape:", trails.A.shape)
print("B shape:", trails.B.shape)


In [ ]:
# Coordinates include years, activities, and flows
trails.A.coords


In [ ]:
# Example entry: year 0, activity 0, product 0
trails.A[
    0,  # first year in the dataset
    0,  # activity 0
    0,  # product 0
]


## 3. Choose LCIA methods

Use `get_lcia_method_names` to list available methods in this package.
We keep methods in a list (order matters for static scores).


In [ ]:
methods = [
    'IPCC 2021 - climate change: total (excl. biogenic CO2) - global warming potential (GWP100)',
    'IPCC 2021 (incl. biogenic CO2) - climate change: total (incl. biogenic CO2) - global warming potential (GWP100)',
]
methods


## 4. Import an Excel inventory

This merges your user inventory into the existing matrices. By default,
exchanges that already exist are replaced, and new activities/flows are added.
Temporal distribution parameters (if provided in the Excel) are preserved.


In [ ]:
# Example Excel inventory
trails.import_excel_inventory('tutorials/DAY 4 - Edges and Trails/assets/trails/lci-pass_cars.xlsx')

### Notes about Excel inventory format

This Excel example file we just imported shows the `bw2io` Excel layout expected by TRAILS:

**Activity section** (field name in column A, value in column B):
- `Activity`: activity name
- `reference product`: reference product name
- `location`: location code
- `unit`: activity unit
- Any other fields

**Exchanges table** (header row starts after the `Exchanges` marker):
- `name`: exchange name
- `amount`: base exchange amount (used when no year-specific columns are provided)
- **Year-specific columns** (e.g., `2020`, `2040`, `2060`): optional numeric columns that set
  year-specific amounts. TRAILS writes these values into the corresponding years in `A`/`B` and
  linearly interpolates between them for intermediate years. Years outside the range are clamped
  to the nearest endpoint value.
- `location`: exchange location (required for technosphere/production linking)
- `categories`: biosphere categories (tuple or `compartment/subcompartment`)
- `unit`: exchange unit
- `type`: `production`, `technosphere`, or `biosphere`
- `reference product`: required for technosphere/production exchanges
- `temporal_distribution`: temporal distribution code
- `temporal_loc`: location parameter (mean/median/mode)
- `temporal_scale`: scale parameter (stddev/sigma)
- `temporal_min` / `temporal_max`: integer offset bounds
- `temporal_amount_source`: `port` (ported amount) or `matrix` (use matrix values)
- `comment`: optional notes


## 5. Explore activities and exchanges

Search for an activity, then inspect its exchanges. Use this to verify that the
imported inventory is linked as expected.


In [ ]:
# Find matching activities
search_activity(trails, "transport, passenger, car,")[-5:]


In [ ]:
# Inspect exchanges for a specific activity index
idx = 39209
trails.print_exchange_table(year=2050, act_idx=39209)

## 6. Temporal routing and graph visualization

Routing shows how demands propagate through time. Use it to understand
long‑term dependencies before solving.


In [ ]:
from trails.plotting import plot_temporal_graph

ref_year = 2050

trails.temporal_routing(
    start_year=ref_year,
    start_act_idx=idx,
    amount=1.0,
    max_depth=3,
    show_progress=True,
    attribute_to_roots=True,
)

plot_temporal_graph(
    trails,
    filename="trails_graph.html",
    notebook=False,
)


## 7. Temporal LCA (full solve)

This computes time‑resolved scores and stores the inventory for downstream
analysis (e.g., climate emulator).


In [ ]:
trails.lca(
    methods=methods,
    show_progress=True,
    compute_score=True,
    store_inventory=True,
)


## 8. Static LCA (single year check)

Static LCA provides a quick single‑year benchmark for the same activity.
Scores are returned in the same order as `methods`.


In [ ]:
trails.static_lca(
    year=ref_year,
    act_idx=idx,
    methods=methods,
)

static_score = trails.static_score
static_score

## 9. Plot temporal scores

By default, TRAILS plots the most important contributors.


In [ ]:
figs = plot_temporal_scores(
    trails=trails,
    stacked=False,
    legend_top_n=7,
    show_flow_contributions=False,
    title="",
    method_label="kg CO₂-eq",
    cumulative=False,
    width=550,
    height=450,
    year_tick=5,
    year_range=(2000, 2100),
    reference_year=ref_year,
    show_cumulative_axis=True,
    static_score=static_score,
    static_score_dash="dot",
    static_score_color="red",
)


In [ ]:
figs[0]

In [ ]:
figs[1]

## 10. Climate emulator (FaIR)

We translate inventory time series into radiative forcing and temperature anomaly.
Results are stored as quantiles (2.5, 25, 50, 75, 97.5).

On some FaIR and SciPy combinations, concurrent per-species FaIR runs can fail
while building the stochastic climate ensemble. For a stable classroom workflow,
the call below keeps `per_species_workers=1`.


In [ ]:
from trails.fair_rf import run_fair_delta_rf

rf = run_fair_delta_rf(
    trails,
    scenario="high-extension",
    scale_target_fraction=0.1,
    per_species_workers=1,
)


### Plot radiative forcing and temperature


In [ ]:
fig = plot_rf(
    trails=trails,
    reference_year=ref_year,
    by="flow",
    year_range=(2000, 2150),
    year_tick=10,
)
fig


In [ ]:
plot_temp(
    trails,
    by="root activity",
    title="Temperature change by root activity",
    method_label="°C",
    year_range=(2000, 2150),
)
